In [0]:
# Read job parameters with defaults for interactive runs
dbutils.widgets.text("BRONZE_TABLE", "bis_dev.bronze_roster.emp_stg_loc", "Bronze Table")
dbutils.widgets.text("ROSTER_TABLE", "None", "Roster Table")
dbutils.widgets.text("SILVER_TABLE", "bis_dev.silver_roster.emp_stg_loc", "Silver Table")
dbutils.widgets.text("QA_TABLE", "bis_dev.bronze_roster.qa_roster", "QA Results Table")

# Get parameter values
BRONZE_TABLE = dbutils.widgets.get("BRONZE_TABLE")
ROSTER_TABLE = dbutils.widgets.get("ROSTER_TABLE")
ROSTER_TABLE = None if ROSTER_TABLE == "None" else ROSTER_TABLE
SILVER_TABLE = dbutils.widgets.get("SILVER_TABLE")
QA_TABLE = dbutils.widgets.get("QA_TABLE")

In [0]:
from datetime import datetime, timezone

QA_RESULT_SCHEMA = ("run_ts timestamp, layer string, file_date date, "
                    "check_name string, severity string, fail_count int, description string")


def run_qa(layer, checks):
    """Run QA checks, persist results to QA_TABLE, and report failures."""
    run_ts = datetime.now(timezone.utc)
    file_date = spark.sql(f"SELECT MAX(File_Date) FROM {BRONZE_TABLE}").collect()[0][0]
    rows = []

    for name, severity, description, sql in checks:
        if sql is None:
            rows.append((run_ts, layer, file_date, name, "SKIPPED", 0, description))
            continue
        try:
            count = int(spark.sql(sql).collect()[0][0] or 0)
            rows.append((run_ts, layer, file_date, name, severity, count, description))
        except Exception as e:
            rows.append((run_ts, layer, file_date, name, "CHECK_FAILED", -1, f"{description} | {e}"))

    df = spark.createDataFrame(rows, QA_RESULT_SCHEMA)
    df.write.mode("append").saveAsTable(QA_TABLE)

    failures = [r for r in rows if r[5] != 0]
    skipped  = [r for r in rows if r[4] == "SKIPPED"]

    for r in failures:
        print(f"[{r[4]}] {r[3]} = {r[5]} — {r[6]}")
    for r in skipped:
        print(f"[SKIPPED] {r[3]} — not configured")

    if not failures:
        print(f"{layer}: {len(rows) - len(skipped)} passed, {len(skipped)} skipped.")

    return df

In [0]:
LATEST_FILE = f"(SELECT MAX(File_Date) FROM {BRONZE_TABLE})"

bronze_checks = [
    (
        "email_matches_roster",
        "ERROR",
        "Rep_Email does not match the roster, or Emp_ID is absent from it",
        None if ROSTER_TABLE is None else
        f"""SELECT COUNT(*) FROM {BRONZE_TABLE} b
            LEFT JOIN {ROSTER_TABLE} r ON b.Emp_ID = r.Emp_ID
            WHERE b.File_Date = {LATEST_FILE}
              AND (r.Emp_ID IS NULL OR lower(b.Rep_Email) <> lower(r.Rep_Email))""",
    ),
    (
        "one_location_per_emp_per_file",
        "WARNING",
        "Emp_ID appears more than once in the same file; Silver keeps the most recently loaded row",
        f"""SELECT COUNT(*) FROM (
              SELECT Emp_ID, File_Name FROM {BRONZE_TABLE}
              WHERE File_Date = {LATEST_FILE}
              GROUP BY Emp_ID, File_Name HAVING COUNT(*) > 1)""",
    ),
    (
        "rescued_data_present",
        "WARNING",
        "Source file contained a new, renamed or extra column",
        f"SELECT COUNT(*) FROM {BRONZE_TABLE} "
        f"WHERE File_Date = {LATEST_FILE} AND _rescued_data IS NOT NULL",
    ),
]

display(run_qa("BRONZE", bronze_checks))

In [0]:
silver_checks = [
    (
        "address_not_parsed",
        "WARNING",
        "Address_1 is blank or Parsing_Error is populated",
        f"SELECT COUNT(*) FROM {SILVER_TABLE} "
        f"WHERE Address_1 IS NULL OR Parsing_Error IS NOT NULL",
    ),
]

display(run_qa("SILVER", silver_checks))